# COMP8240 · SanText final report: privacy experiment and LLM judge (notebook C)

Run this **after notebook A/B**; it reads A/B's outputs from `MyDrive/COMP8240_final`.

| Step | What runs | Report section | Time |
|---|---|---|---|
| 0–1 | Setup, inputs (SST-2, QNLI, Dreaddit, your reviews, A/B's sanitized files) | | 5 min |
| 2 | Statistics of the original datasets, measured from the files | §2 (QNLI detail) | 3 min |
| 3 | **LLM judge** (open-weights model, runs on this GPU): meaning kept, disclosure, leftover disclosure; human-check sheet | §5 | 25 min |
| 4 | SanText with **BERT embeddings** on SST-2, ε = 8–20, computed block by block (the authors' version needs ~19 GB) | §3 | 5 min |
| 5 | **Mask token inference attack**, as released and with its input bug fixed | §3 | 5 min |
| 6 | Check that the block-wise sanitizer matches the authors' code (GloVe, ε = 1–3) | §3 | 5 min |
| 7 | Fine-tuning on the BERT-sanitized SST-2 at ε = 16 (the paper's Table 3 and privacy claim) | §3 | 15 min |
| 8 | Summary tables and one zip | | 2 min |

QNLI is sanitized in the background (block by block) while the GPU works, to show the memory problem is an implementation issue.

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save, then **Runtime → Run all**, and approve Google Drive. Keep the tab open and click into it every 20–30 minutes.

**If Colab disconnects:** Runtime → Run all again. Finished steps are skipped.

**When it finishes**, Step 8 downloads `notebook_c_outputs.zip` (also saved in `MyDrive/COMP8240_final/c`). Upload it in the chat, together with `human_check.xlsx` once you've filled it in (about 20 minutes, any time).

In [ ]:
# Step 0 · Setup (about 3 minutes)
import os, sys, re, json, time, glob, shutil, subprocess, hashlib, platform, zipfile
BASE = "/content/workc"
for d in ("data", "logs", "results", "results_c", "results_c/attack", "results_c/judge", "san_bert", "san_glove_check", "san_qnli"):
    os.makedirs(f"{BASE}/{d}", exist_ok=True)
os.chdir(BASE)

import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Runtime > Change runtime type > T4 GPU > Save, then Runtime > Run all again.")

from google.colab import drive
try:
    drive.mount("/content/drive")
except Exception as e:
    raise RuntimeError("Google Drive did not connect. Run this cell again; in the Google window choose the account that has COMP8240_update, click 'Select all' on the permissions screen, then Continue.") from e
OLD = "/content/drive/MyDrive/COMP8240_update"   # 9 Oct: your review dataset and the first runs
NEW = "/content/drive/MyDrive/COMP8240_final"    # notebook A/B
C = f"{NEW}/c"                                   # this notebook
os.makedirs(C, exist_ok=True)


def backup(*paths):
    for p in paths:
        if not os.path.exists(p):
            continue
        dest = os.path.join(C, os.path.relpath(p, BASE))
        if os.path.isdir(p):
            shutil.copytree(p, dest, dirs_exist_ok=True)
        else:
            os.makedirs(os.path.dirname(dest), exist_ok=True)
            shutil.copy2(p, dest)


def run(cmd, log):
    """Run a command, write its output to logs/<log>, show the last lines, and back the log up."""
    with open(f"{BASE}/logs/{log}", "w") as lf:
        code = subprocess.run(cmd, cwd=BASE, stdout=lf, stderr=subprocess.STDOUT).returncode
    lines = open(f"{BASE}/logs/{log}", errors="ignore").read().splitlines()
    print(("ok" if code == 0 else f"!! exit {code}") + f" | {log}\n   " + "\n   ".join(lines[-4:]), flush=True)
    backup(f"{BASE}/logs/{log}")
    return code


restored = []
for sub in ("results", "results_c", "san_bert", "san_glove_check", "san_qnli", "logs"):
    if os.path.isdir(f"{C}/{sub}"):
        shutil.copytree(f"{C}/{sub}", f"{BASE}/{sub}", dirs_exist_ok=True)
        restored.append(sub)
print("restored from Drive:", ", ".join(restored) or "nothing (first session)")

SANTEXT_COMMIT = "9c35f6e"
if not os.path.isdir(f"{BASE}/SanText"):
    !git clone -q https://github.com/xiangyue9607/SanText.git {BASE}/SanText
!cd {BASE}/SanText && git checkout -q {SANTEXT_COMMIT} && git log -1 --format="SanText code: commit %h (%cd)" --date=short
!pip -q install bitsandbytes 2>&1 | tail -1

import numpy, scipy, sklearn, spacy, transformers
env = {"python": platform.python_version(), "torch": torch.__version__, "transformers": transformers.__version__,
       "spacy": spacy.__version__, "numpy": numpy.__version__, "scipy": scipy.__version__, "scikit-learn": sklearn.__version__,
       "gpu": torch.cuda.get_device_name(0), "cpu_cores": os.cpu_count(),
       "ram_gb": round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9, 1)}
json.dump(env, open(f"{BASE}/results_c/environment.json", "w"), indent=2)
print(json.dumps(env, indent=2))

### Scripts and the annotation labels (written to disk; nothing to do here)
- `finetune_sanitized.py`: the fine-tuning script used on 9 Oct and in A/B, unchanged.
- `santext_chunked.py`: SanText with the authors' own probability function, a block of rows at a time. Tested against the authors' code: the rows are identical (max. difference ≈ 1e-10), and the replacements follow the same distribution.
- `mask_attack.py`: the paper's mask token inference attack, as released (inputs swapped) and fixed.
- `llm_judge.py`, `build_judge_items.py`: the LLM judge and its items. `analyse_c.py`: summary tables.
- `annotation_labels.csv`: the labels of your 100 annotated reviews (annotator A, gold). There's no review text; the text is looked up from your Drive.

In [ ]:
%%writefile finetune_sanitized.py
#!/usr/bin/env python3
"""
finetune_sanitized.py: fine-tune BERT on (sanitized) text and report dev-set utility.

Why this script exists
  SanText's released fine-tuning code (run_glue.py + dataset_glue.py) imports
  transformers 3.3.1 module paths such as `transformers.tokenization_bart`,
  which were removed in transformers 4.0, so it no longer runs on a current
  install. This script re-implements the same fine-tuning step with the paper's
  settings (Yue et al., 2021, Sec. 5.1, SST-2/QNLI):
      bert-base-uncased, max_seq_length 128, batch size 64, learning rate 2e-5,
      3 epochs, AdamW without weight decay, linear decay without warmup,
      gradient clipping 1.0, seed 42.
  One deliberate difference: each batch is padded to its longest example rather
  than to 128 tokens. Padding is masked out of attention, so the model computes
  the same thing, but short-sentence data such as SST-2 trains several times faster.

Input
  --data_dir must contain train.tsv and dev.tsv with a header row, as written by
  run_SanText.py (or the original GLUE files):
    single sentence (SST-2 style):  sentence <TAB> label
    sentence pair  (QNLI style):    index <TAB> text_a <TAB> text_b <TAB> label
  The format is detected from the header (2 columns = single, 4 or more = pair).

Output
  results/<run_name>/metrics.json         settings, environment, per-epoch and final dev metrics
  results/<run_name>/dev_predictions.tsv  gold and predicted label for every dev row

Examples
  # 1-minute smoke test (run this first on a new machine)
  python finetune_sanitized.py --data_dir data/SST-2 --max_train_samples 512 --epochs 1 --run_name smoke_test
  # unsanitized baseline (paper: 0.9251 accuracy)
  python finetune_sanitized.py --data_dir data/SST-2 --run_name SST-2_unsanitized --fp16
  # sanitized data written by run_SanText.py
  python finetune_sanitized.py --data_dir output_SanText_glove/SST-2/eps_3.00 --fp16
  # long texts (e.g. Dreaddit): longer inputs, same effective batch of 64 in two halves
  python finetune_sanitized.py --data_dir new_data/dreaddit --max_seq_length 256 --grad_accum 2 --fp16
"""
import argparse
import json
import math
import os
import platform
import random
import time
from contextlib import nullcontext

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import numpy as np
import torch
from torch.utils.data import DataLoader

import transformers
from transformers import AutoModelForSequenceClassification, AutoTokenizer


def parse_args():
    p = argparse.ArgumentParser(description="Fine-tune BERT on (sanitized) TSV data and report dev-set utility.")
    p.add_argument("--data_dir", required=True, help="folder containing train.tsv and dev.tsv")
    p.add_argument("--run_name", default=None, help="results folder name (default: built from data_dir)")
    p.add_argument("--output_root", default="results")
    p.add_argument("--model_name", default="bert-base-uncased")
    p.add_argument("--max_seq_length", type=int, default=128)
    p.add_argument("--batch_size", type=int, default=64, help="effective batch size per optimizer step")
    p.add_argument("--grad_accum", type=int, default=1,
                   help="split each batch into this many micro-batches to save GPU memory (effective batch unchanged)")
    p.add_argument("--eval_batch_size", type=int, default=128)
    p.add_argument("--learning_rate", type=float, default=2e-5)
    p.add_argument("--epochs", type=int, default=3)
    p.add_argument("--warmup_ratio", type=float, default=0.0)
    p.add_argument("--weight_decay", type=float, default=0.0)
    p.add_argument("--max_grad_norm", type=float, default=1.0)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--fp16", action="store_true", help="mixed precision on GPU (faster on T4/P100-class GPUs)")
    p.add_argument("--format", choices=["auto", "single", "pair"], default="auto")
    p.add_argument("--max_train_samples", type=int, default=None, help="use only the first N training rows (smoke tests)")
    p.add_argument("--log_every", type=int, default=200, help="print training loss every N optimizer steps")
    p.add_argument("--save_model", action="store_true", help="also save the fine-tuned model (about 440 MB)")
    args = p.parse_args()
    if args.batch_size % args.grad_accum != 0:
        p.error("--batch_size must be divisible by --grad_accum")
    return args


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def read_tsv(path, fmt):
    """Return (format, texts_a, texts_b or None, labels, skipped_rows)."""
    texts_a, texts_b, labels, skipped = [], [], [], 0
    with open(path, encoding="utf-8") as f:
        header = f.readline().rstrip("\n").split("\t")
        if fmt == "auto":
            fmt = "single" if len(header) == 2 else "pair"
        for line in f:
            cols = line.rstrip("\n").split("\t")
            if fmt == "single":
                text, label = "\t".join(cols[:-1]).strip(), cols[-1].strip()
                if len(cols) < 2 or not text or not label:
                    skipped += 1
                    continue
                texts_a.append(text)
            else:
                if len(cols) < 4 or not cols[-1].strip():
                    skipped += 1
                    continue
                texts_a.append(cols[1])
                texts_b.append(cols[2])
                label = cols[-1].strip()
            labels.append(label)
    return fmt, texts_a, (texts_b if fmt == "pair" else None), labels, skipped


def build_label_map(labels):
    def key(x):
        return (0, int(x), "") if x.lstrip("-").isdigit() else (1, 0, x)
    return {lab: i for i, lab in enumerate(sorted(set(labels), key=key))}


def make_examples(texts_a, texts_b, labels, label_map, split_name):
    unknown = sorted(set(labels) - set(label_map))
    if unknown:
        raise ValueError(f"{split_name}.tsv has labels not seen in train.tsv: {unknown[:5]}")
    b = texts_b if texts_b is not None else [None] * len(texts_a)
    return [(a, bb, label_map[lab]) for a, bb, lab in zip(texts_a, b, labels)]


def make_collate(tokenizer, max_len):
    def collate(batch):
        a = [ex[0] for ex in batch]
        b = [ex[1] for ex in batch] if batch[0][1] is not None else None
        enc = tokenizer(a, b, truncation=True, max_length=max_len, padding=True, return_tensors="pt")
        enc["labels"] = torch.tensor([ex[2] for ex in batch], dtype=torch.long)
        return enc
    return collate


def make_scaler(enabled):
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError):  # older torch
        return torch.cuda.amp.GradScaler(enabled=enabled)


def amp(enabled):
    """Mixed-precision context on GPU; a no-op otherwise."""
    return torch.autocast(device_type="cuda", dtype=torch.float16) if enabled else nullcontext()


def compute_metrics(gold, pred, n_labels):
    gold, pred = np.asarray(gold), np.asarray(pred)
    f1s = []
    for c in range(n_labels):
        tp = int(((pred == c) & (gold == c)).sum())
        fp = int(((pred == c) & (gold != c)).sum())
        fn = int(((pred != c) & (gold == c)).sum())
        prec = tp / (tp + fp) if tp + fp else 0.0
        rec = tp / (tp + fn) if tp + fn else 0.0
        f1s.append(2 * prec * rec / (prec + rec) if prec + rec else 0.0)
    out = {"accuracy": float((gold == pred).mean()), "macro_f1": float(np.mean(f1s))}
    if n_labels == 2:
        out["f1_label_index_1"] = float(f1s[1])  # positive class for SST-2, "stress" for Dreaddit
    return out


@torch.no_grad()
def evaluate(model, loader, device, use_fp16, n_labels):
    model.eval()
    preds, golds = [], []
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        with amp(use_fp16):
            logits = model(**{k: v for k, v in batch.items() if k != "labels"}).logits
        preds.extend(logits.argmax(dim=-1).tolist())
        golds.extend(batch["labels"].tolist())
    model.train()
    return compute_metrics(golds, preds, n_labels), preds


def main():
    args = parse_args()
    set_seed(args.seed)
    t_start = time.time()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_fp16 = args.fp16 and device.type == "cuda"
    if args.fp16 and not use_fp16:
        print("note: --fp16 ignored because no GPU is available")
    gpu_name = torch.cuda.get_device_name(0) if device.type == "cuda" else None

    # ---- data -------------------------------------------------------------
    fmt, tr_a, tr_b, tr_y, tr_skip = read_tsv(os.path.join(args.data_dir, "train.tsv"), args.format)
    _, dv_a, dv_b, dv_y, dv_skip = read_tsv(os.path.join(args.data_dir, "dev.tsv"), fmt)
    if args.max_train_samples:
        tr_a, tr_y = tr_a[: args.max_train_samples], tr_y[: args.max_train_samples]
        tr_b = tr_b[: args.max_train_samples] if tr_b is not None else None
    label_map = build_label_map(tr_y)
    n_labels = len(label_map)
    train_ex = make_examples(tr_a, tr_b, tr_y, label_map, "train")
    dev_ex = make_examples(dv_a, dv_b, dv_y, label_map, "dev")
    print(f"data: {args.data_dir} | format={fmt} | train={len(train_ex)} dev={len(dev_ex)} "
          f"| skipped rows train={tr_skip} dev={dv_skip} | labels={label_map}")

    tokenizer = AutoTokenizer.from_pretrained(args.model_name)
    collate = make_collate(tokenizer, args.max_seq_length)
    micro_bs = args.batch_size // args.grad_accum
    gen = torch.Generator().manual_seed(args.seed)
    train_loader = DataLoader(train_ex, batch_size=micro_bs, shuffle=True, collate_fn=collate, generator=gen)
    dev_loader = DataLoader(dev_ex, batch_size=args.eval_batch_size, shuffle=False, collate_fn=collate)

    # ---- model, optimizer, schedule ------------------------------------------
    model = AutoModelForSequenceClassification.from_pretrained(args.model_name, num_labels=n_labels).to(device)
    no_decay = ("bias", "LayerNorm.weight")
    groups = [
        {"params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
         "weight_decay": args.weight_decay},
        {"params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
         "weight_decay": 0.0},
    ]
    optimizer = torch.optim.AdamW(groups, lr=args.learning_rate, eps=1e-8)
    steps_per_epoch = math.ceil(len(train_loader) / args.grad_accum)
    total_steps = steps_per_epoch * args.epochs
    warmup_steps = int(args.warmup_ratio * total_steps)

    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        return max(0.0, (total_steps - step) / max(1, total_steps - warmup_steps))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler = make_scaler(use_fp16)

    # ---- training -------------------------------------------------------------
    print(f"training on {gpu_name or 'CPU'} | {args.epochs} epochs x {steps_per_epoch} steps | "
          f"batch {args.batch_size} (micro-batch {micro_bs}) | lr {args.learning_rate} | fp16={use_fp16}")
    history, global_step = [], 0
    model.train()
    for epoch in range(1, args.epochs + 1):
        running_loss, n_loss = 0.0, 0
        optimizer.zero_grad(set_to_none=True)
        for i, batch in enumerate(train_loader, start=1):
            batch = {k: v.to(device) for k, v in batch.items()}
            with amp(use_fp16):
                loss = model(**batch).loss / args.grad_accum
            scaler.scale(loss).backward()
            running_loss += loss.item() * args.grad_accum
            n_loss += 1
            if i % args.grad_accum == 0 or i == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), args.max_grad_norm)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
                global_step += 1
                if global_step % args.log_every == 0:
                    print(f"  epoch {epoch} step {global_step}/{total_steps} | loss {running_loss / n_loss:.4f} "
                          f"| lr {scheduler.get_last_lr()[0]:.2e} | {(time.time() - t_start) / 60:.1f} min")
        dev_metrics, preds = evaluate(model, dev_loader, device, use_fp16, n_labels)
        history.append({"epoch": epoch, "train_loss": running_loss / max(1, n_loss), "dev": dev_metrics,
                        "elapsed_min": round((time.time() - t_start) / 60, 2)})
        print(f"epoch {epoch} done | train loss {running_loss / max(1, n_loss):.4f} | dev {dev_metrics}")

    # ---- results (final = after the last epoch, as the original run_glue.py reports) ----
    final_metrics = history[-1]["dev"]
    run_name = args.run_name or "_".join(
        [p for p in os.path.normpath(os.path.abspath(args.data_dir)).split(os.sep) if p][-3:])
    out_dir = os.path.join(args.output_root, run_name)
    os.makedirs(out_dir, exist_ok=True)
    id2label = {v: k for k, v in label_map.items()}
    with open(os.path.join(out_dir, "dev_predictions.tsv"), "w", encoding="utf-8") as f:
        f.write("text_a\ttext_b\tgold\tpred\n")
        for (a, b, gold), pred in zip(dev_ex, preds):
            f.write(f"{a}\t{b or ''}\t{id2label[gold]}\t{id2label[pred]}\n")
    runtime_min = round((time.time() - t_start) / 60, 2)
    report = {
        "run_name": run_name,
        "data_dir": os.path.abspath(args.data_dir),
        "format": fmt,
        "label_map": label_map,
        "n_train": len(train_ex),
        "n_dev": len(dev_ex),
        "skipped_rows": {"train": tr_skip, "dev": dv_skip},
        "settings": vars(args),
        "environment": {
            "python": platform.python_version(),
            "torch": torch.__version__,
            "transformers": transformers.__version__,
            "device": gpu_name or "CPU",
            "platform": platform.platform(),
        },
        "epochs": history,
        "final_dev": final_metrics,
        "runtime_min": runtime_min,
    }
    with open(os.path.join(out_dir, "metrics.json"), "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2)
    if args.save_model:
        model.save_pretrained(os.path.join(out_dir, "model"))
        tokenizer.save_pretrained(os.path.join(out_dir, "model"))
    summary = " ".join(f"{k}={v:.4f}" for k, v in final_metrics.items())
    print(f"RESULT {run_name} | {summary} | n_dev={len(dev_ex)} | {runtime_min} min on {gpu_name or 'CPU'}")
    print(f"saved to {out_dir}/")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile santext_chunked.py
#!/usr/bin/env python3
"""
santext_chunked.py: SanText (Yue et al., 2021) computed a block of probability rows at a time.

The authors' run_SanText.py builds the whole |V| x |V| probability matrix
    P[w, w'] = softmax over w' of ( -eps/2 * ||phi(w) - phi(w')|| )
in memory and holds about five copies of it at its peak: about 5.3 GB for SST-2 with GloVe
(14,283 words with vectors), about 19 GB with BERT's 30,522-token vocabulary and about 160 GB
for QNLI with GloVe (88,159 words). Colab has 12.7 GB, so the paper's BERT-embedding experiments
and QNLI cannot run as released. This script samples from exactly the same rows, computed with
the authors' own cal_probability function, but only for words that occur in the data and only a
block of rows at a time.

Differences from run_SanText.py, all deliberate:
  * One seeded NumPy generator in a single process draws every replacement, so --seed fixes the
    output. (In the released code, forked worker processes reuse copies of one random state.)
  * SanText only (every word is sensitive), which is all the paper's privacy experiments use.
Everything else follows run_SanText.py: the vocabulary (the authors' get_vocab_SST2 /
get_vocab_QNLI for GloVe; every BERT token for BERT), the tokenizers (spaCy's blank English for
GloVe, bert-base-uncased WordPiece for BERT), the order of the embedding rows, the uniform
replacement of words that have no GloVe vector, and the output format (tokens joined by single
spaces, then the label; QNLI as index, question, sentence, label).

Usage
  python santext_chunked.py --task SST-2 --embedding bert --epsilon 14 --data_dir data/SST-2 \
      --output_dir san_bert/SST-2 --santext_dir SanText
  python santext_chunked.py --task SST-2 --embedding glove --glove data/glove.filtered.txt \
      --epsilon 3 --data_dir data/SST-2 --output_dir san_glove/SST-2 --santext_dir SanText --check_rows 2000
Output: <output_dir>/eps_<eps>/{train.tsv, dev.tsv, meta.json}
"""
import argparse
import json
import os
import resource
import sys
import time

import numpy as np


def parse_args():
    p = argparse.ArgumentParser(description="SanText with the authors' probabilities, computed in blocks of rows.")
    p.add_argument("--task", choices=["SST-2", "QNLI"], default="SST-2",
                   help="input format: SST-2 (sentence, label) or QNLI (index, question, sentence, label)")
    p.add_argument("--embedding", choices=["glove", "bert"], required=True)
    p.add_argument("--glove", default=None, help="GloVe text file (full or filtered); needed for --embedding glove")
    p.add_argument("--bert_model", default="bert-base-uncased")
    p.add_argument("--epsilon", type=float, required=True)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--data_dir", required=True, help="folder with train.tsv and dev.tsv")
    p.add_argument("--output_dir", required=True, help="results go to <output_dir>/eps_<epsilon>/")
    p.add_argument("--santext_dir", required=True, help="the authors' repository (utils.py, SanText.py)")
    p.add_argument("--block", type=int, default=1024, help="probability rows computed at once")
    p.add_argument("--splits", default="train,dev", help="which files to sanitize")
    p.add_argument("--check_rows", type=int, default=0,
                   help="compare this many rows with the authors' full matrix (GloVe vocabularies up to 20,000 words)")
    return p.parse_args()


# ----------------------------------------------------------------------------- vocabulary and embeddings
def load_glove(data_dir, task, glove_path, santext_dir):
    """Exactly run_SanText.py's construction for --embedding_type glove (SanText: all words sensitive)."""
    from utils import get_vocab_SST2, get_vocab_QNLI, word_normalize  # the authors' functions
    from spacy.lang.en import English
    tokenizer = English()
    get_vocab = {"SST-2": get_vocab_SST2, "QNLI": get_vocab_QNLI}[task]
    vocab = get_vocab(data_dir, tokenizer, tokenizer_type="word")
    all_words = [k for k, _ in vocab.most_common()]
    word2id, embs = {}, []
    with open(glove_path) as f:
        line = f.readline().rstrip().split(" ")
        if len(line) != 2:
            f.seek(0)
        for row in f:
            content = row.rstrip().split(" ")
            cur_word = word_normalize(content[0])
            if cur_word in vocab and cur_word not in word2id:
                word2id[cur_word] = len(embs)
                embs.append([float(x) for x in content[1:]])
    emb = np.array(embs, dtype="f")
    id2word = [None] * len(word2id)
    for w, i in word2id.items():
        id2word[i] = w

    def tokenize(text):
        return [t.text for t in tokenizer(text)]

    return tokenize, all_words, word2id, id2word, emb, len(vocab)


def load_bert_embeddings(model_name):
    """BERT's input embedding matrix (one row per token id) and its tokenizer."""
    from transformers import AutoModelForMaskedLM, AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForMaskedLM.from_pretrained(model_name)
    emb = model.get_input_embeddings().weight.detach().cpu().numpy().astype("f")
    del model
    return tokenizer, emb


def load_bert(model_name):
    """run_SanText.py with --embedding_type bert: every token of BERT's vocabulary is in V (and sensitive),
    rows in token-id order, docs tokenized into word pieces."""
    tokenizer, emb_matrix = load_bert_embeddings(model_name)
    items = sorted(tokenizer.get_vocab().items(), key=lambda kv: kv[1])
    word2id = {tok: k for k, (tok, _) in enumerate(items)}
    id2word = [tok for tok, _ in items]
    emb = emb_matrix[[i for _, i in items]].astype("f")
    return tokenizer.tokenize, id2word, word2id, id2word, emb, len(id2word)


# ----------------------------------------------------------------------------- data
def read_split(path, task):
    """As run_SanText.py reads it: header kept; SST-2 -> (text, label); QNLI -> (q, s, label) per row."""
    with open(path, encoding="utf-8") as f:
        header = next(f)
        rows = []
        for line in f:
            content = line.strip().split("\t")
            if task == "SST-2":
                rows.append((content[0], int(content[1])))
            else:
                rows.append((content[1], content[2], content[-1]))
    return header, rows


def write_split(path, header, rows, san_docs, task):
    with open(path, "w", encoding="utf-8") as out:
        out.write(header)
        if task == "SST-2":
            for (_, label), doc in zip(rows, san_docs):
                out.write(" ".join(doc) + "\t" + str(label) + "\n")
        else:
            for i, (_, _, label) in enumerate(rows):
                out.write(str(i) + "\t" + " ".join(san_docs[2 * i]) + "\t" + " ".join(san_docs[2 * i + 1])
                          + "\t" + str(label) + "\n")


# ----------------------------------------------------------------------------- sampling
def sanitize_docs(docs, word2id, id2word, all_words, emb, eps, rng, block, cal_probability):
    """Replace every token: in-vocabulary tokens from their row of the authors' probability matrix,
    tokens without a vector uniformly from the whole vocabulary (as run_SanText.py does)."""
    lengths = [len(d) for d in docs]
    flat = [t for d in docs for t in d]
    rows = np.fromiter((word2id.get(t, -1) for t in flat), dtype=np.int64, count=len(flat))
    out = np.empty(len(flat), dtype=object)
    oov = np.flatnonzero(rows < 0)
    if len(oov):
        out[oov] = np.asarray(all_words, dtype=object)[rng.integers(0, len(all_words), size=len(oov))]
    inv = np.flatnonzero(rows >= 0)
    order = inv[np.argsort(rows[inv], kind="stable")]
    uniq, starts, counts = np.unique(rows[order], return_index=True, return_counts=True)
    id2word_arr = np.asarray(id2word, dtype=object)
    for b in range(0, len(uniq), block):
        ids = uniq[b:b + block]
        prob = cal_probability(emb[ids], emb, eps)  # the authors' function, on a block of rows
        for k in range(len(ids)):
            s, c = starts[b + k], counts[b + k]
            cdf = np.cumsum(prob[k], dtype=np.float64)
            idx = np.searchsorted(cdf, rng.random(c) * cdf[-1], side="right")
            out[order[s:s + c]] = id2word_arr[np.minimum(idx, len(cdf) - 1)]
        del prob
    san, pos = [], 0
    for n in lengths:
        san.append(list(out[pos:pos + n]))
        pos += n
    unchanged = float(np.mean([a == b for a, b in zip(flat, out)])) if flat else None
    return san, {"tokens": len(flat), "without_vector": int(len(oov)), "rows_computed": int(len(uniq)),
                 "share_unchanged": unchanged}


def check_rows(emb, eps, n, block, rng, cal_probability):
    """Rows computed in blocks vs the authors' full matrix (small vocabularies), or block-size invariance."""
    sample = np.sort(rng.choice(len(emb), size=min(n, len(emb)), replace=False))
    if len(emb) <= 20000:
        reference = cal_probability(emb, emb, eps)[sample]
        how = "full |V| x |V| matrix, as run_SanText.py computes it"
    else:
        reference = np.vstack([cal_probability(emb[sample[i:i + 97]], emb, eps) for i in range(0, len(sample), 97)])
        how = "blocks of 97 rows (block-size invariance; the full matrix does not fit in memory)"
    mine = np.vstack([cal_probability(emb[sample[i:i + block]], emb, eps) for i in range(0, len(sample), block)])
    diff = np.abs(reference.astype(np.float64) - mine.astype(np.float64))
    return {"rows_checked": int(len(sample)), "reference": how, "max_abs_diff": float(diff.max()),
            "rows_identical": int(np.sum(np.all(reference == mine, axis=1))),
            "argmax_identical": int(np.sum(reference.argmax(1) == mine.argmax(1)))}


def main():
    args = parse_args()
    sys.path.insert(0, os.path.abspath(args.santext_dir))
    from SanText import cal_probability  # the authors' distance + softmax, unchanged

    t0 = time.time()
    if args.embedding == "glove":
        if not args.glove:
            raise SystemExit("--glove is required with --embedding glove")
        tokenize, all_words, word2id, id2word, emb, vocab_size = load_glove(args.data_dir, args.task, args.glove,
                                                                            args.santext_dir)
    else:
        tokenize, all_words, word2id, id2word, emb, vocab_size = load_bert(args.bert_model)
    print(f"vocabulary {vocab_size:,} | rows with vectors {len(emb):,} x {emb.shape[1]} | "
          f"one full matrix would be {len(emb) ** 2 * 4 / 1e9:.2f} GB", flush=True)

    rng = np.random.default_rng(args.seed)
    out_dir = os.path.join(args.output_dir, "eps_%.2f" % args.epsilon)
    os.makedirs(out_dir, exist_ok=True)
    meta = {"task": args.task, "embedding": args.embedding, "epsilon": args.epsilon, "seed": args.seed,
            "block": args.block, "vocabulary": vocab_size, "rows_with_vectors": int(len(emb)),
            "full_matrix_gb": round(len(emb) ** 2 * 4 / 1e9, 2), "splits": {}}
    for split in args.splits.split(","):
        header, rows = read_split(os.path.join(args.data_dir, f"{split}.tsv"), args.task)
        texts = [r[0] for r in rows] if args.task == "SST-2" else [t for r in rows for t in (r[0], r[1])]
        docs = [tokenize(t) for t in texts]
        san, info = sanitize_docs(docs, word2id, id2word, all_words, emb, args.epsilon, rng, args.block, cal_probability)
        write_split(os.path.join(out_dir, f"{split}.tsv"), header, rows, san, args.task)
        meta["splits"][split] = {"rows": len(rows), **info}
        print(f"{split}: {len(rows):,} rows | {info['tokens']:,} tokens | unchanged {info['share_unchanged']:.4f} | "
              f"without vector {info['without_vector']:,}", flush=True)
    if args.check_rows:
        meta["check"] = check_rows(emb, args.epsilon, args.check_rows, args.block, np.random.default_rng(0), cal_probability)
        print("row check:", json.dumps(meta["check"]), flush=True)
    meta["seconds"] = round(time.time() - t0, 1)
    meta["peak_memory_gb"] = round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss * 1024 / 1e9, 2)
    with open(os.path.join(out_dir, "meta.json"), "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)
    print(f"done in {meta['seconds']} s, peak memory {meta['peak_memory_gb']} GB -> {out_dir}", flush=True)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile mask_attack.py
#!/usr/bin/env python3
"""
mask_attack.py: the mask token inference attack of Yue et al. (2021, Sec. 5.3), as released and as fixed.

For every word piece j of a sanitized document, replace piece j with [MASK], let a pretrained BERT
masked language model predict it, and count a success when the top-1 prediction is the ORIGINAL
piece at j. Defense rate = 1 - success rate (higher = more private).

Two variants:
  released  What the authors' bert_inference_token.py computes in its own environment
            (transformers 3.3.1). Its TensorDataset holds (input_ids, token_type_ids,
            attention_mask, labels), but the model call passes attention_mask=batch[1] (the
            token-type ids: all 0 for one sentence) and token_type_ids=batch[2] (the attention
            mask). In 3.3.1 an all-zero mask adds the same -10000 to every attention score, which
            changes nothing, so the model attends to the padding as well; and every real token gets
            the second-segment (B) embedding. Inputs are padded to 64 word pieces as in the script
            (--max_seq_length 64); positions beyond the 62 that fit lose their [MASK] and are
            skipped and counted here (the released script would stop with a shape error).
            We reproduce this by passing an all-ones attention mask and the real attention mask as
            token_type_ids, which is the same computation.
  fixed     The same attack with the inputs in their intended places and no truncation.

Input: the original dev.tsv (SST-2 format) and a sanitized dev.tsv whose text is word pieces joined
by single spaces (as run_SanText.py and santext_chunked.py write it with --embedding bert), or
--sanitized none to attack the unsanitized text (epsilon = infinity).

Usage
  python mask_attack.py --original data/SST-2/dev.tsv --sanitized san_bert/SST-2/eps_14.00/dev.tsv \
      --name eps14 --out_dir results_c/attack [--variants released,fixed]
"""
import argparse
import json
import os
import time


def read_texts(path, sanitized):
    with open(path, encoding="utf-8") as f:
        next(f)
        if sanitized:
            return [line.rstrip("\n").rsplit("\t", 1)[0] for line in f]
        return [line.strip().split("\t")[0] for line in f]  # as bert_inference_token.py reads it


def build_instances(orig_docs, san_docs, variant, to_ids, cls_id, sep_id, mask_id, max_len=64):
    """One instance per word piece: (input ids, real length, mask position, label id, kept, doc, pos).
    Returns the instances and the number of positions skipped because truncation removed the mask."""
    instances, skipped = [], 0
    for d, (o, s) in enumerate(zip(orig_docs, san_docs)):
        if len(o) != len(s):
            raise ValueError(f"document {d}: {len(o)} original pieces but {len(s)} sanitized pieces")
        san_ids, orig_ids = to_ids(s), to_ids(o)
        for j in range(len(s)):
            ids = list(san_ids)
            ids[j] = mask_id
            if variant == "released":
                if j >= max_len - 2:
                    skipped += 1
                    continue
                ids = ids[: max_len - 2]
            else:
                ids = ids[:510]
                if j >= 510:
                    skipped += 1
                    continue
            seq = [cls_id] + ids + [sep_id]
            instances.append((seq, len(seq), j + 1, orig_ids[j], s[j] == o[j], d, j))
    return instances, skipped


def make_batch(batch, variant, pad_id, max_len, torch, device):
    """Tensors for one batch. released: pad to max_len, attention over everything, segment ids = real mask."""
    width = max_len if variant == "released" else max(b[1] for b in batch)
    ids = torch.full((len(batch), width), pad_id, dtype=torch.long)
    real = torch.zeros((len(batch), width), dtype=torch.long)
    for k, b in enumerate(batch):
        ids[k, : b[1]] = torch.tensor(b[0], dtype=torch.long)
        real[k, : b[1]] = 1
    if variant == "released":
        attention, segments = torch.ones_like(real), real
    else:
        attention, segments = real, torch.zeros_like(real)
    return ids.to(device), attention.to(device), segments.to(device)


def predict_at(model, ids, attention, segments, pos, torch, device):
    """Top-1 prediction at each row's [MASK] position. Runs BERT once and applies the masked-LM head
    only at the masked positions (same numbers as the full head, far less work)."""
    rows = torch.arange(ids.shape[0], device=device)
    body, head = getattr(model, "bert", None), getattr(model, "cls", None)
    if body is not None and head is not None:
        hidden = body(input_ids=ids, attention_mask=attention, token_type_ids=segments).last_hidden_state
        return head(hidden[rows, pos]).argmax(-1).tolist()
    logits = model(input_ids=ids, attention_mask=attention, token_type_ids=segments).logits
    return logits[rows, pos].argmax(-1).tolist()


def run_attack(instances, variant, model, pad_id, max_len, batch_size, torch, device, autocast=None):
    correct = []
    order = sorted(range(len(instances)), key=lambda i: instances[i][1]) if variant == "fixed" else range(len(instances))
    order = list(order)
    with torch.no_grad():
        for b in range(0, len(order), batch_size):
            batch = [instances[i] for i in order[b:b + batch_size]]
            ids, attention, segments = make_batch(batch, variant, pad_id, max_len, torch, device)
            pos = torch.tensor([x[2] for x in batch], device=device)
            if autocast is not None:
                with autocast():
                    pred = predict_at(model, ids, attention, segments, pos, torch, device)
            else:
                pred = predict_at(model, ids, attention, segments, pos, torch, device)
            correct.extend((order[b + k], int(p == x[3])) for k, (p, x) in enumerate(zip(pred, batch)))
    hits = [0] * len(instances)
    for i, c in correct:
        hits[i] = c
    return hits


def summarise(instances, hits, skipped):
    n = len(instances)
    kept = [h for inst, h in zip(instances, hits) if inst[4]]
    changed = [h for inst, h in zip(instances, hits) if not inst[4]]
    rate = lambda xs: round(1 - sum(xs) / len(xs), 4) if xs else None
    return {"positions": n, "skipped_truncated": skipped, "attack_success": round(sum(hits) / n, 4) if n else None,
            "defense_rate": rate(hits), "positions_kept_by_sanitizer": len(kept), "defense_rate_kept": rate(kept),
            "positions_changed_by_sanitizer": len(changed), "defense_rate_changed": rate(changed)}


def main():
    ap = argparse.ArgumentParser(description="Mask token inference attack, as released and fixed.")
    ap.add_argument("--original", required=True, help="original dev.tsv (SST-2 format)")
    ap.add_argument("--sanitized", required=True, help="sanitized dev.tsv, or 'none' for unsanitized text")
    ap.add_argument("--name", required=True)
    ap.add_argument("--out_dir", required=True)
    ap.add_argument("--model", default="bert-base-uncased")
    ap.add_argument("--variants", default="released,fixed")
    ap.add_argument("--max_seq_length", type=int, default=64, help="released script's default")
    ap.add_argument("--batch_size", type=int, default=256)
    ap.add_argument("--fp32", action="store_true", help="disable mixed precision on the GPU")
    args = ap.parse_args()

    import torch
    from transformers import AutoModelForMaskedLM, AutoTokenizer
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tok = AutoTokenizer.from_pretrained(args.model)
    model = AutoModelForMaskedLM.from_pretrained(args.model).to(device).eval()
    autocast = None
    if device.type == "cuda" and not args.fp32:
        autocast = lambda: torch.autocast(device_type="cuda", dtype=torch.float16)

    orig_docs = [tok.tokenize(t) for t in read_texts(args.original, sanitized=False)]
    if args.sanitized == "none":
        san_docs = [list(d) for d in orig_docs]
    else:
        san_docs = [t.split(" ") if t else [] for t in read_texts(args.sanitized, sanitized=True)]
    to_ids = tok.convert_tokens_to_ids
    report = {"name": args.name, "sanitized": args.sanitized, "model": args.model, "documents": len(orig_docs),
              "max_seq_length_released": args.max_seq_length, "mixed_precision": autocast is not None, "variants": {}}
    for variant in args.variants.split(","):
        t0 = time.time()
        inst, skipped = build_instances(orig_docs, san_docs, variant, to_ids, tok.cls_token_id, tok.sep_token_id,
                                        tok.mask_token_id, args.max_seq_length)
        hits = run_attack(inst, variant, model, tok.pad_token_id, args.max_seq_length, args.batch_size, torch, device,
                          autocast)
        report["variants"][variant] = {**summarise(inst, hits, skipped), "seconds": round(time.time() - t0, 1)}
        print(f"{args.name} [{variant}]: " + json.dumps(report["variants"][variant]), flush=True)
    os.makedirs(args.out_dir, exist_ok=True)
    with open(os.path.join(args.out_dir, f"{args.name}.json"), "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile llm_judge.py
#!/usr/bin/env python3
"""
llm_judge.py: LLM-as-judge for sanitized text, run locally with an open-weights model.

The model never writes free text. For each prompt we read its probabilities for the allowed answer
tokens at the first position of its reply, so every judgment is deterministic, cheap and comes with
a confidence:
  meaning     original + sanitized text -> how well is the meaning preserved? 1-5
              score = expected value over the probabilities of the tokens "1".."5"
  disclosure  one review -> does it reveal personal information about its writer? yes/no
              (the same definition as the human annotation guidelines)
  residual    sanitized review + the information its original disclosed -> is it still revealed? yes/no
              p_yes = P(yes) / (P(yes) + P(no)), summing "yes"/"Yes" and "no"/"No"
format_mass records how much of the model's probability falls on the allowed answer tokens at all
(close to 1 means the model is following the answer format).

Nothing leaves the machine: the model is downloaded once and run on the local GPU.

Usage
  python llm_judge.py --items judge_items.jsonl --out judge_results.jsonl
      [--model Qwen/Qwen2.5-7B-Instruct --fallback Qwen/Qwen2.5-3B-Instruct --batch_size 8]
Resumable: items already in --out are skipped.
"""
import argparse
import json
import os
import time

PROMPT_VERSION = "v1"
SYSTEM = ("You are a careful annotator in a research study on text privacy. "
          "Follow the instructions exactly and answer with a single token.")

MEANING = """Text A is an original text. Text B was made from Text A by replacing some of its words with other words.
How well does Text B preserve the meaning of Text A?
1 = unrelated or unintelligible
2 = mostly lost; only fragments survive
3 = the topic survives, but important details are lost or changed
4 = mostly the same meaning; minor details are lost
5 = the same meaning

Text A: <<<{a}>>>
Text B: <<<{b}>>>

Answer with one digit from 1 to 5."""

DISCLOSURE = """Does the following app review reveal personal information about its writer, or about another identifiable person, that they might not want linked to them?
Personal information means:
- HEALTH: a condition, symptom, medication, treatment, pregnancy or mental health
- FINANCE: amounts, debts, loans, income, account problems, or which bank they use
- LOCATION: suburb, address, workplace or branch
- IDENTITY: a name, age, family member or job
- CONTACT: an email, phone number, account or reference number
Opinions about the app, its features or its price, and generic use ("I use it daily", "my meds", "my account") do not count.

Review: <<<{a}>>>

Answer yes or no."""

RESIDUAL = """An original app review disclosed this personal information about its writer: {info}.
The text below was made from that review by replacing some of its words with other words.
Does the text below still reveal that information, or enough of it that a reader could infer it?

Text: <<<{b}>>>

Answer yes or no."""


def clip_words(text, n):
    words = (text or "").split()
    return " ".join(words[:n]) + (" [...]" if len(words) > n else "")


def build_prompt(item, max_words):
    a, b = clip_words(item.get("a"), max_words), clip_words(item.get("b"), max_words)
    if item["task"] == "meaning":
        return MEANING.format(a=a, b=b)
    if item["task"] == "disclosure":
        return DISCLOSURE.format(a=a)
    if item["task"] == "residual":
        return RESIDUAL.format(info=item["info"], b=b)
    raise ValueError(item["task"])


def answer_token_ids(tokenizer):
    def first_id(s):
        ids = tokenizer.encode(s, add_special_tokens=False)
        return ids[0] if ids else None
    digits = {d: first_id(d) for d in "12345"}
    yes = sorted({i for i in (first_id("yes"), first_id("Yes")) if i is not None})
    no = sorted({i for i in (first_id("no"), first_id("No")) if i is not None})
    return digits, yes, no


def load_model(model_id, quantize, torch):
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(model_id)
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    kwargs = {"device_map": "auto"}
    if quantize:
        from transformers import BitsAndBytesConfig
        kwargs["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                           bnb_4bit_compute_dtype=torch.float16)
    try:
        model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16, **kwargs)
    except TypeError:  # older transformers use torch_dtype
        model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, **kwargs)
    return tok, model.eval()


def decoder_and_head(model):
    """The transformer body and the output layer, so only the last position's logits are computed."""
    try:
        decoder = model.get_decoder()
    except Exception:
        decoder = None
    if decoder is None:
        decoder = getattr(model, "model", None) or model.base_model
    head = model.get_output_embeddings()
    if head is None:
        head = model.lm_head
    return decoder, head


def last_position_logits(decoder, head, enc):
    """Logits of the next token only (the full logits tensor would not fit for a batch)."""
    attention = enc["attention_mask"]
    position_ids = (attention.cumsum(-1) - 1).clamp(min=0)
    hidden = decoder(input_ids=enc["input_ids"], attention_mask=attention,
                     position_ids=position_ids).last_hidden_state[:, -1, :]
    return head(hidden).float()


def main():
    ap = argparse.ArgumentParser(description="Local LLM-as-judge for sanitized text.")
    ap.add_argument("--items", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--model", default="Qwen/Qwen2.5-7B-Instruct")
    ap.add_argument("--quantize", default="4bit", choices=["4bit", "none"])
    ap.add_argument("--fallback", default="Qwen/Qwen2.5-3B-Instruct", help="used without quantization if the first model fails to load")
    ap.add_argument("--batch_size", type=int, default=8)
    ap.add_argument("--max_words", type=int, default=200, help="each text is cut to this many words")
    args = ap.parse_args()

    import torch
    with open(args.items, encoding="utf-8") as f:
        items = [json.loads(l) for l in f if l.strip()]
    done = set()
    if os.path.exists(args.out):
        with open(args.out, encoding="utf-8") as f:
            done = {json.loads(l)["id"] for l in f if l.strip()}
    todo = [it for it in items if it["id"] not in done]
    print(f"{len(items)} items, {len(done)} already judged, {len(todo)} to go", flush=True)
    if not todo:
        return

    model_id, quant = args.model, args.quantize == "4bit"
    try:
        tok, model = load_model(model_id, quant, torch)
    except Exception as e:  # e.g. bitsandbytes unavailable
        print(f"could not load {model_id} ({type(e).__name__}: {e}); falling back to {args.fallback}", flush=True)
        model_id, quant = args.fallback, False
        tok, model = load_model(model_id, quant, torch)
    model_label = model_id + (" (4-bit NF4)" if quant else " (fp16)")
    digits, yes_ids, no_ids = answer_token_ids(tok)
    print(f"judge: {model_label} | digit tokens {digits} | yes {yes_ids} | no {no_ids}", flush=True)
    device = next(model.parameters()).device
    decoder, head = decoder_and_head(model)

    # longest prompts first, so memory problems show up at once
    todo.sort(key=lambda it: -len(build_prompt(it, args.max_words)))
    t0 = time.time()
    with open(args.out, "a", encoding="utf-8") as out, torch.no_grad():
        for b in range(0, len(todo), args.batch_size):
            batch = todo[b:b + args.batch_size]
            texts = [tok.apply_chat_template([{"role": "system", "content": SYSTEM},
                                              {"role": "user", "content": build_prompt(it, args.max_words)}],
                                             tokenize=False, add_generation_prompt=True) for it in batch]
            enc = tok(texts, return_tensors="pt", padding=True).to(device)
            probs = torch.softmax(last_position_logits(decoder, head, enc), dim=-1).cpu()
            for it, p in zip(batch, probs):
                rec = dict(it)
                rec.update({"judge_model": model_label, "prompt_version": PROMPT_VERSION,
                            "top_token": tok.decode([int(p.argmax())])})
                if it["task"] == "meaning":
                    pd = {d: float(p[i]) for d, i in digits.items()}
                    mass = sum(pd.values())
                    rec.update({"p_digits": {d: v / mass for d, v in pd.items()}, "format_mass": mass,
                                "expected": sum(int(d) * v for d, v in pd.items()) / mass,
                                "answer": int(max(pd, key=pd.get))})
                else:
                    py, pn = float(p[yes_ids].sum()), float(p[no_ids].sum())
                    rec.update({"p_yes": py / (py + pn), "format_mass": py + pn, "answer": "yes" if py > pn else "no"})
                for k in ("a", "b", "info"):
                    rec.pop(k, None)  # keep the results file free of the texts
                out.write(json.dumps(rec, ensure_ascii=False) + "\n")
            out.flush()
            if (b // args.batch_size) % 25 == 0:
                print(f"  {b + len(batch)}/{len(todo)} judged ({(time.time() - t0) / 60:.1f} min)", flush=True)
    print(f"finished in {(time.time() - t0) / 60:.1f} min with {model_label}", flush=True)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile build_judge_items.py
#!/usr/bin/env python3
"""
build_judge_items.py: the items the LLM judge rates, and a blind sheet for the human check.

Items (one JSON object per line in judge_items.jsonl):
  meaning     original vs sanitized dev text, for SST-2, Dreaddit and the app reviews:
              100 texts per dataset (for the reviews, the 100 annotated ones) x SanText and SanText+
              at eps = 1, 2, 3 (seed 42), plus 20 identical pairs per dataset as a sanity check
  disclosure  the 100 annotated reviews, original and all six sanitized versions
  residual    the reviews whose original discloses something (gold label), six sanitized versions,
              plus the originals as a sanity check
human_check.xlsx: 24 meaning and 12 residual items, chosen at random within strata, without the judge's
answers; human_check_key.csv maps each row back to its item.

Usage
  python build_judge_items.py --config judge_inputs.json --out_dir judge
"""
import argparse
import collections
import csv
import hashlib
import json
import os
import random
import re

METHODS = (("SanText", "SanText"), ("SanText_plus", "SanText+"))
EPS = (1.0, 2.0, 3.0)


def read_original(path):
    with open(path, encoding="utf-8") as f:
        next(f)
        return [line.strip().split("\t")[0] for line in f]


def read_sanitized(path):
    with open(path, encoding="utf-8") as f:
        next(f)
        return [line.rstrip("\n").rsplit("\t", 1)[0] for line in f]


def norm(text):
    text = text.replace("’", "'").replace("‘", "'").replace("“", '"').replace("”", '"')
    return re.sub(r"\s+", " ", text).strip().lower()


def review_texts(labels, reviews_clean, annotation_sheet):
    """ann_id -> review text: review_key = sha1(review_id)[:12] in reviews_clean.jsonl, else the sheet's text."""
    by_key = {}
    if reviews_clean and os.path.exists(reviews_clean):
        with open(reviews_clean, encoding="utf-8") as f:
            for line in f:
                r = json.loads(line)
                key = hashlib.sha1((r.get("review_id") or r["text"]).encode("utf-8")).hexdigest()[:12]
                by_key[key] = r["text"]
    by_id = {}
    if annotation_sheet and os.path.exists(annotation_sheet):
        with open(annotation_sheet, encoding="utf-8-sig", newline="") as f:
            by_id = {int(r["ann_id"]): r["text"] for r in csv.DictReader(f) if r.get("ann_id")}
    out, how = {}, collections.Counter()
    for r in labels:
        i = int(r["ann_id"])
        if r["review_key"] in by_key:
            out[i] = by_key[r["review_key"]]
            how["review_key"] += 1
        elif i in by_id:
            out[i] = by_id[i]
            how["annotation_sheet"] += 1
    return out, dict(how)


def align(texts, dev_texts):
    """ann_id -> row in the dev (test) split: exact text, then normalised text."""
    exact = {t: k for k, t in reversed(list(enumerate(dev_texts)))}
    normed = {norm(t): k for k, t in reversed(list(enumerate(dev_texts)))}
    out, how = {}, collections.Counter()
    for i, t in texts.items():
        if t in exact:
            out[i] = exact[t]
            how["exact"] += 1
        elif norm(t) in normed:
            out[i] = normed[norm(t)]
            how["normalised"] += 1
        else:
            how["not_found"] += 1
    return out, dict(how)


def sanitized_path(san_root, ds, method, eps, old_sst2=None):
    """Where notebook A/B wrote the seed-42 output; for SST-2 SanText, fall back to the 9 Oct run."""
    sub = os.path.join(san_root, ds, method, "seed_42", "eps_%.2f" % eps)
    if method == "SanText_plus":
        sub = os.path.join(sub, "sword_0.90_p_0.30")
    p = os.path.join(sub, "dev.tsv")
    if os.path.exists(p):
        return p, "notebook A/B"
    if ds == "sst2" and method == "SanText" and old_sst2:
        q = os.path.join(old_sst2, "eps_%.2f" % eps, "dev.tsv")
        if os.path.exists(q):
            return q, "9 Oct run"
    return None, None


def info_string(words, cats):
    w = [x.strip() for x in words.split(";") if x.strip()]
    c = [x.strip().lower() for x in cats.split(";") if x.strip()]
    return "; ".join(f'"{x}" ({c[k] if k < len(c) else c[0]})' for k, x in enumerate(w))


def write_human_sheet(rows, path):
    from openpyxl import Workbook
    from openpyxl.styles import Alignment, Font, PatternFill
    from openpyxl.worksheet.datavalidation import DataValidation
    wb = Workbook()
    how = wb.active
    how.title = "How to"
    lines = [
        "Human check of the LLM judge (about 20 minutes)",
        "",
        "Each row on the 'Check' sheet shows an original text (A) and a version made from it by SanText (B).",
        "Fill only the 'your answer' column. Don't look anything up; go with your first reading.",
        "",
        "Rows marked MEANING: how well does B preserve the meaning of A?",
        "  1 = unrelated or unintelligible",
        "  2 = mostly lost; only fragments survive",
        "  3 = the topic survives, but important details are lost or changed",
        "  4 = mostly the same meaning; minor details are lost",
        "  5 = the same meaning",
        "",
        "Rows marked STILL REVEALS: the 'information' column says what the original review disclosed.",
        "  Answer yes if B still reveals that information, or enough of it that a reader could infer it; otherwise no.",
        "",
        "Save the file and upload it in the chat when you're done.",
    ]
    for k, line in enumerate(lines, start=1):
        how.cell(row=k, column=1, value=line).font = Font(bold=(k == 1), size=12 if k == 1 else 11)
    how.column_dimensions["A"].width = 120
    ws = wb.create_sheet("Check")
    header = ["check_id", "task", "A: original", "B: sanitized", "information (STILL REVEALS rows)", "your answer"]
    ws.append(header)
    for c in ws[1]:
        c.font = Font(bold=True)
        c.fill = PatternFill("solid", fgColor="DDE7F0")
    for r in rows:
        ws.append([r["check_id"], r["task_label"], r["a"], r["b"], r.get("info", ""), None])
    for col, width in zip("ABCDEF", (10, 15, 60, 60, 35, 14)):
        ws.column_dimensions[col].width = width
    for row in ws.iter_rows(min_row=2):
        for c in row:
            c.alignment = Alignment(wrap_text=True, vertical="top")
    n = len(rows) + 1
    meaning_rows = [k + 2 for k, r in enumerate(rows) if r["task"] == "meaning"]
    residual_rows = [k + 2 for k, r in enumerate(rows) if r["task"] == "residual"]
    if meaning_rows:
        dv = DataValidation(type="whole", operator="between", formula1="1", formula2="5", allow_blank=True)
        dv.error = "Enter a whole number from 1 to 5"
        ws.add_data_validation(dv)
        for k in meaning_rows:
            dv.add(f"F{k}")
    if residual_rows:
        dv2 = DataValidation(type="list", formula1='"yes,no"', allow_blank=True)
        ws.add_data_validation(dv2)
        for k in residual_rows:
            dv2.add(f"F{k}")
    ws.freeze_panes = "C2"
    ws.auto_filter.ref = f"A1:F{n}"
    wb.active = 1
    wb.save(path)


def main():
    ap = argparse.ArgumentParser(description="Build the LLM-judge items and the human-check sheet.")
    ap.add_argument("--config", required=True)
    ap.add_argument("--out_dir", required=True)
    ap.add_argument("--per_dataset", type=int, default=100)
    ap.add_argument("--sanity", type=int, default=20)
    ap.add_argument("--human_meaning_per_dataset", type=int, default=8)
    ap.add_argument("--human_residual", type=int, default=12)
    ap.add_argument("--seed", type=int, default=0)
    args = ap.parse_args()
    with open(args.config, encoding="utf-8") as f:
        cfg = json.load(f)
    os.makedirs(args.out_dir, exist_ok=True)
    rng = random.Random(args.seed)
    report = {"sources": {}, "counts": {}}

    with open(cfg["labels"], encoding="utf-8") as f:
        labels = list(csv.DictReader(f))
    texts, how_text = review_texts(labels, cfg.get("reviews_clean"), cfg.get("annotation_sheet"))
    report["review_texts_found"] = how_text

    originals = {ds: read_original(p) for ds, p in cfg["original_dev"].items() if p and os.path.exists(p)}
    versions = {}  # (ds, method, eps) -> sanitized dev texts
    for ds in originals:
        for method, _ in METHODS:
            for e in EPS:
                p, src = sanitized_path(cfg["san_root"], ds, method, e, cfg.get("old_sst2"))
                if p:
                    san = read_sanitized(p)
                    if len(san) != len(originals[ds]):
                        raise ValueError(f"{p}: {len(san)} rows, original has {len(originals[ds])}")
                    versions[(ds, method, e)] = san
                    report["sources"][f"{ds}/{method}/eps{e:g}"] = src
    items = []

    # which dev rows each dataset uses
    picks = {}
    if "reviews" in originals:
        rowmap, how_align = align(texts, originals["reviews"])
        report["review_alignment"] = how_align
        picks["reviews"] = {f"r{i:03d}": k for i, k in sorted(rowmap.items())}
    for ds in originals:
        if ds != "reviews":
            idx = sorted(random.Random(f"{args.seed}-{ds}").sample(range(len(originals[ds])),
                                                                    min(args.per_dataset, len(originals[ds]))))
            picks[ds] = {f"{ds[0]}{k:04d}": k for k in idx}

    # meaning
    for ds, chosen in picks.items():
        for key, row in chosen.items():
            for method, mlabel in METHODS:
                for e in EPS:
                    if (ds, method, e) in versions:
                        items.append({"id": f"meaning|{ds}|{key}|{method}|{e:g}", "task": "meaning", "dataset": ds,
                                      "text_key": key, "row": row, "method": mlabel, "epsilon": e,
                                      "a": originals[ds][row], "b": versions[(ds, method, e)][row]})
        for key in list(chosen)[: args.sanity]:
            row = chosen[key]
            items.append({"id": f"meaning|{ds}|{key}|identical|0", "task": "meaning", "dataset": ds, "text_key": key,
                          "row": row, "method": "identical", "epsilon": None,
                          "a": originals[ds][row], "b": originals[ds][row]})

    # disclosure and residual (annotated reviews)
    lab = {f"r{int(r['ann_id']):03d}": r for r in labels}
    for key, row in picks.get("reviews", {}).items():
        r = lab[key]
        base = {"dataset": "reviews", "text_key": key, "row": row, "gold": r["gold_disclosure"],
                "A": r["A_disclosure"], "category": r["category"]}
        items.append({"id": f"disclosure|{key}|original|0", "task": "disclosure", "method": "original", "epsilon": None,
                      "a": originals["reviews"][row], **base})
        for method, mlabel in METHODS:
            for e in EPS:
                if ("reviews", method, e) in versions:
                    items.append({"id": f"disclosure|{key}|{method}|{e:g}", "task": "disclosure", "method": mlabel,
                                  "epsilon": e, "a": versions[("reviews", method, e)][row], **base})
        if r["gold_disclosure"] == "yes":
            info = info_string(r["gold_words"], r["gold_categories"])
            items.append({"id": f"residual|{key}|original|0", "task": "residual", "method": "original", "epsilon": None,
                          "info": info, "b": originals["reviews"][row], **base})
            for method, mlabel in METHODS:
                for e in EPS:
                    if ("reviews", method, e) in versions:
                        items.append({"id": f"residual|{key}|{method}|{e:g}", "task": "residual", "method": mlabel,
                                      "epsilon": e, "info": info, "a": originals["reviews"][row],
                                      "b": versions[("reviews", method, e)][row], **base})

    with open(os.path.join(args.out_dir, "judge_items.jsonl"), "w", encoding="utf-8") as f:
        for it in items:
            f.write(json.dumps(it, ensure_ascii=False) + "\n")
    report["counts"] = dict(collections.Counter(f"{it['task']}/{it['dataset']}" for it in items))

    # blind human-check sheet
    human = []
    for ds in picks:
        pool = [it for it in items if it["task"] == "meaning" and it["dataset"] == ds and it["method"] != "identical"]
        by_combo = collections.defaultdict(list)
        for it in pool:
            by_combo[(it["method"], it["epsilon"])].append(it)
        combos = sorted(by_combo)
        used_texts, chosen = set(), []
        k = 0
        while len(chosen) < args.human_meaning_per_dataset and any(by_combo.values()):
            combo = combos[k % len(combos)]
            k += 1
            cands = [it for it in by_combo[combo] if it["text_key"] not in used_texts]
            if not cands:
                if k > 10 * len(combos):
                    break
                continue
            it = rng.choice(cands)
            used_texts.add(it["text_key"])
            chosen.append(it)
        human += chosen
    res_pool = [it for it in items if it["task"] == "residual" and it["method"] != "original"]
    for mlabel in ("SanText", "SanText+"):
        sub = [it for it in res_pool if it["method"] == mlabel]
        human += rng.sample(sub, min(args.human_residual // 2, len(sub)))
    rng.shuffle(human)
    rows, key_rows = [], []
    for n, it in enumerate(human, start=1):
        rows.append({"check_id": n, "task": it["task"], "task_label": "MEANING" if it["task"] == "meaning" else "STILL REVEALS",
                     # STILL REVEALS rows show what the judge sees: the information and B, not the original
                     "a": it["a"] if it["task"] == "meaning" else "(not shown: judge from the information column and B)",
                     "b": it["b"], "info": it.get("info", "")})
        key_rows.append({"check_id": n, "item_id": it["id"]})
    if rows:
        write_human_sheet(rows, os.path.join(args.out_dir, "human_check.xlsx"))
        with open(os.path.join(args.out_dir, "human_check_key.csv"), "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=["check_id", "item_id"])
            w.writeheader()
            w.writerows(key_rows)
    report["human_check_rows"] = dict(collections.Counter(r["task"] for r in rows))
    with open(os.path.join(args.out_dir, "items_report.json"), "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2)
    print(json.dumps(report, indent=2))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile analyse_c.py
#!/usr/bin/env python3
"""
analyse_c.py: summary tables for notebook C (privacy experiment, sanitizer check, LLM judge).

Reads results_c/ (written by the notebook) and writes results_c/summary_c.md plus CSV tables:
  datasets.json          -> original-dataset statistics (SST-2, QNLI) as measured from the files
  glove_check.csv        -> block-wise sanitizer vs the authors' code (rows; unchanged-token rates)
  bert_santext.csv       -> SanText with BERT embeddings on SST-2: unchanged tokens, utility vs the
                            paper's Table 3, defense rate of the mask token attack (released vs fixed)
  judge_*.csv            -> LLM-judge tables (meaning, disclosure, residual disclosure)
Usage: python analyse_c.py --root results_c --plan_c plan_c.json
"""
import argparse
import collections
import csv
import json
import os
import statistics

TABLE3 = {12.0: 0.6084, 14.0: 0.7548, 16.0: 0.8698}  # Yue et al. (2021), Table 3, SST-2, "Original"
UNSANITIZED_PAPER = 0.9251


def load_json(path):
    try:
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    except (OSError, ValueError):
        return None


def kappa(a, b):
    n = len(a)
    if not n:
        return None
    po = sum(x == y for x, y in zip(a, b)) / n
    ca, cb = collections.Counter(a), collections.Counter(b)
    pe = sum(ca[k] * cb[k] for k in set(a) | set(b)) / (n * n)
    return (po - pe) / (1 - pe) if pe < 1 else 1.0


def fmt(x, nd=3):
    return "–" if x is None else f"{x:.{nd}f}"


def write_csv(path, rows):
    if not rows:
        return
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]))
        w.writeheader()
        w.writerows(rows)


def unchanged_share(orig_path, san_path, tokenizer):
    with open(orig_path, encoding="utf-8") as f:
        next(f)
        orig = [[t.text for t in tokenizer(line.strip().split("\t")[0])] for line in f]
    with open(san_path, encoding="utf-8") as f:
        next(f)
        san = [line.rstrip("\n").rsplit("\t", 1)[0].split(" ") for line in f]
    same = tot = 0
    for o, s in zip(orig, san):
        if len(o) == len(s):
            tot += len(o)
            same += sum(a == b for a, b in zip(o, s))
    return same / tot if tot else None


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", required=True)
    ap.add_argument("--plan_c", required=True)
    args = ap.parse_args()
    plan = load_json(args.plan_c)
    R = args.root
    md = ["# Notebook C: summary", ""]
    env = load_json(os.path.join(R, "environment.json"))
    if env:
        md += ["Environment: " + ", ".join(f"{k} {v}" for k, v in env.items()), ""]

    # ---------------------------------------------------------------- original datasets
    ds = load_json(os.path.join(R, "datasets.json"))
    if ds:
        md += ["## Original datasets, measured from the files", ""]
        for name, d in ds.items():
            md.append(f"**{name}**: " + "; ".join(f"{k}: {v}" for k, v in d.items() if k != "examples"))
            for ex in d.get("examples", []):
                md.append(f"  - `{ex}`")
            md.append("")

    # ---------------------------------------------------------------- block-wise sanitizer vs the authors'
    rows = []
    try:
        from spacy.lang.en import English
        tok = English()
    except Exception:
        tok = None
    for e in plan.get("glove_check_eps", []):
        meta = load_json(os.path.join(plan["glove_check_dir"], "eps_%.2f" % e, "meta.json"))
        if not meta:
            continue
        row = {"epsilon": e, "rows_checked": meta.get("check", {}).get("rows_checked"),
               "max_abs_diff": meta.get("check", {}).get("max_abs_diff"),
               "rows_identical": meta.get("check", {}).get("rows_identical"),
               "chunked_unchanged_dev": meta["splits"]["dev"]["share_unchanged"],
               "chunked_peak_gb": meta.get("peak_memory_gb"), "chunked_seconds": meta.get("seconds")}
        if tok:
            for label, folder in plan.get("authors_glove_dirs", {}).items():
                p = os.path.join(folder, "eps_%.2f" % e, "dev.tsv")
                if os.path.exists(p):
                    row[f"authors_unchanged_dev ({label})"] = unchanged_share(plan["sst2_dev"], p, tok)
            row["chunked_unchanged_dev_recount"] = unchanged_share(
                plan["sst2_dev"], os.path.join(plan["glove_check_dir"], "eps_%.2f" % e, "dev.tsv"), tok)
        rows.append(row)
    write_csv(os.path.join(R, "glove_check.csv"), rows)
    if rows:
        md += ["## Block-wise sanitizer vs the authors' code (SST-2, GloVe, SanText, seed 42)", "",
               "| ε | Rows checked | Max abs. difference | Rows bit-identical | Unchanged tokens (dev), block-wise | Unchanged tokens (dev), authors' runs |",
               "|---|---|---|---|---|---|"]
        for r in rows:
            authors = ", ".join(f"{fmt(v, 4)} ({k.split('(')[1][:-1]})" for k, v in r.items() if k.startswith("authors_") and v is not None)
            md.append(f"| {r['epsilon']:g} | {r['rows_checked']} | {r['max_abs_diff']:.1e} | {r['rows_identical']} | "
                      f"{fmt(r.get('chunked_unchanged_dev_recount') or r['chunked_unchanged_dev'], 4)} | {authors or '–'} |")
        md.append("")

    # ---------------------------------------------------------------- BERT-embedding SanText: utility and attack
    rows = []
    base_attack = load_json(os.path.join(R, "attack", "unsanitized.json"))
    unsan = load_json(os.path.join(plan["results_dir"], plan["unsanitized_run"], "metrics.json"))
    for e in [None] + plan.get("bert_eps", []):
        if e is None:
            meta, metrics, attack = None, unsan, base_attack
        else:
            meta = load_json(os.path.join(plan["bert_dir"], "eps_%.2f" % e, "meta.json"))
            metrics = load_json(os.path.join(plan["results_dir"], f"sst2__SanText_bert__eps{e:.2f}__seed42", "metrics.json"))
            attack = load_json(os.path.join(R, "attack", f"eps{e:g}.json"))
        v = (attack or {}).get("variants", {})
        rows.append({"epsilon": "∞ (unsanitized)" if e is None else e,
                     "unchanged_dev": None if not meta else meta["splits"]["dev"]["share_unchanged"],
                     "accuracy": None if not metrics else metrics["final_dev"]["accuracy"],
                     "paper_accuracy": UNSANITIZED_PAPER if e is None else TABLE3.get(e),
                     "defense_released": v.get("released", {}).get("defense_rate"),
                     "defense_fixed": v.get("fixed", {}).get("defense_rate"),
                     "skipped_released": v.get("released", {}).get("skipped_truncated"),
                     "positions": v.get("fixed", {}).get("positions"),
                     "seconds": None if not meta else meta.get("seconds"),
                     "peak_gb": None if not meta else meta.get("peak_memory_gb")})
    write_csv(os.path.join(R, "bert_santext.csv"), rows)
    md += ["## SanText with BERT embeddings on SST-2 (dev): utility and the mask token inference attack", "",
           "| ε | Tokens unchanged | Accuracy (ours) | Accuracy (paper, Table 3) | Defense rate, attack as released | Defense rate, attack fixed | Positions (skipped as released) | Sanitizer s / peak GB |",
           "|---|---|---|---|---|---|---|---|"]
    for r in rows:
        md.append(f"| {r['epsilon'] if isinstance(r['epsilon'], str) else format(r['epsilon'], 'g')} | "
                  f"{fmt(r['unchanged_dev'])} | {fmt(r['accuracy'], 4)} | {fmt(r['paper_accuracy'], 4)} | "
                  f"{fmt(r['defense_released'])} | {fmt(r['defense_fixed'])} | {r['positions'] or '–'} ({r['skipped_released'] if r['skipped_released'] is not None else '–'}) | "
                  f"{r['seconds'] if r['seconds'] is not None else '–'} / {r['peak_gb'] if r['peak_gb'] is not None else '–'} |")
    md.append("")

    # ---------------------------------------------------------------- LLM judge
    jr = os.path.join(R, "judge", "judge_results.jsonl")
    if os.path.exists(jr):
        with open(jr, encoding="utf-8") as f:
            res = [json.loads(l) for l in f if l.strip()]
        model = res[0]["judge_model"] if res else "?"
        md += [f"## LLM judge ({model}, prompt {res[0]['prompt_version'] if res else '?'})", ""]
        mass = collections.defaultdict(list)
        for r in res:
            mass[r["task"]].append(r["format_mass"])
        md.append("Probability on the allowed answer tokens (format compliance): " +
                  ", ".join(f"{k} {statistics.mean(v):.3f}" for k, v in mass.items()))
        md.append("")
        # meaning
        g = collections.defaultdict(list)
        for r in res:
            if r["task"] == "meaning":
                g[(r["dataset"], r["method"], r["epsilon"])].append(r["expected"])
        mrows = [{"dataset": d, "method": m, "epsilon": e, "n": len(v), "mean": statistics.mean(v),
                  "sd": statistics.stdev(v) if len(v) > 1 else None} for (d, m, e), v in sorted(g.items(), key=lambda kv: (kv[0][0], kv[0][1], kv[0][2] or 0))]
        write_csv(os.path.join(R, "judge_meaning.csv"), mrows)
        md += ["### Meaning preserved (1–5, expected value)", "", "| Dataset | Method | ε | n | Mean | SD |", "|---|---|---|---|---|---|"]
        md += [f"| {r['dataset']} | {r['method']} | {'–' if r['epsilon'] is None else format(r['epsilon'], 'g')} | {r['n']} | {r['mean']:.2f} | {fmt(r['sd'], 2)} |" for r in mrows]
        md.append("")
        # disclosure on originals: agreement
        orig = [r for r in res if r["task"] == "disclosure" and r["method"] == "original"]
        if orig:
            judge = [r["answer"] for r in orig]
            gold = [r["gold"] for r in orig]
            a = [r["A"] for r in orig]
            md += ["### Disclosure on the 100 original reviews (the judge as a third annotator)", "",
                   f"- vs gold (adjudicated): agreement {sum(x == y for x, y in zip(judge, gold)) / len(orig):.2f}, κ {fmt(kappa(judge, gold), 2)}",
                   f"- vs annotator A: agreement {sum(x == y for x, y in zip(judge, a)) / len(orig):.2f}, κ {fmt(kappa(judge, a), 2)}",
                   f"- judged as disclosing: {sum(x == 'yes' for x in judge)} of {len(orig)} (gold: {sum(x == 'yes' for x in gold)})", ""]
        # disclosure after sanitization
        g = collections.defaultdict(lambda: [0, 0, 0, 0])
        for r in res:
            if r["task"] == "disclosure":
                k = (r["method"], r["epsilon"])
                if r["gold"] == "yes":
                    g[k][0] += 1
                    g[k][1] += r["answer"] == "yes"
                else:
                    g[k][2] += 1
                    g[k][3] += r["answer"] == "yes"
        drows = [{"method": m, "epsilon": e, "gold_yes": v[0], "still_judged_disclosing": v[1],
                  "share_still_disclosing": v[1] / v[0] if v[0] else None, "gold_no": v[2],
                  "judged_disclosing_spuriously": v[3], "share_spurious": v[3] / v[2] if v[2] else None}
                 for (m, e), v in sorted(g.items(), key=lambda kv: (kv[0][0] != "original", kv[0][0], kv[0][1] or 0))]
        write_csv(os.path.join(R, "judge_disclosure.csv"), drows)
        md += ["### Is a disclosure still judged present after sanitization? (reviews)", "",
               "| Version | ε | Gold-disclosing reviews still judged disclosing | Non-disclosing reviews judged disclosing |", "|---|---|---|---|"]
        md += [f"| {r['method']} | {'–' if r['epsilon'] is None else format(r['epsilon'], 'g')} | {r['still_judged_disclosing']}/{r['gold_yes']} ({fmt(r['share_still_disclosing'], 2)}) | "
               f"{r['judged_disclosing_spuriously']}/{r['gold_no']} ({fmt(r['share_spurious'], 2)}) |" for r in drows]
        md.append("")
        # residual
        g = collections.defaultdict(list)
        for r in res:
            if r["task"] == "residual":
                g[(r["method"], r["epsilon"])].append(r)
        rrows = [{"method": m, "epsilon": e, "n": len(v), "still_reveals": sum(x["answer"] == "yes" for x in v),
                  "share": sum(x["answer"] == "yes" for x in v) / len(v), "mean_p_yes": statistics.mean(x["p_yes"] for x in v)}
                 for (m, e), v in sorted(g.items(), key=lambda kv: (kv[0][0] != "original", kv[0][0], kv[0][1] or 0))]
        write_csv(os.path.join(R, "judge_residual.csv"), rrows)
        md += ["### Targeted check: does the sanitized review still reveal the flagged information?", "",
               "| Version | ε | n | Still reveals | Mean P(yes) |", "|---|---|---|---|---|"]
        md += [f"| {r['method']} | {'–' if r['epsilon'] is None else format(r['epsilon'], 'g')} | {r['n']} | {r['still_reveals']} ({r['share']:.2f}) | {r['mean_p_yes']:.2f} |" for r in rrows]
        md.append("")
    with open(os.path.join(R, "summary_c.md"), "w", encoding="utf-8") as f:
        f.write("\n".join(md) + "\n")
    print("\n".join(md))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile annotation_labels.csv
ann_id,review_key,category,A_disclosure,gold_disclosure,gold_words,gold_categories,resolution
1,a976f32825fc,health,yes,yes,antidepressants,HEALTH,agreed
2,7b82d5caecb5,finance,no,no,,,agreed
3,cca519ff0422,finance,yes,yes,$50; $100; my payid; my account,FINANCE; FINANCE; FINANCE; FINANCE,agreed
4,4e41a9d4f089,health,no,no,,,agreed
5,255c97c1d604,finance,no,no,,,agreed
6,602f91fcf62b,finance,no,no,,,agreed
7,03a49232483f,health,no,no,,,agreed
8,a2f830f6f6fa,health,yes,no,,,resolved by guideline
9,71734f54c9c2,health,no,no,,,agreed
10,efa15a518d98,health,no,no,,,agreed
11,7450a2747fa7,finance,yes,yes,[NAME]; Shellhabour Branch,IDENTITY; LOCATION,agreed
12,3201b9f72e2b,finance,no,no,,,agreed
13,20c372fd13f0,health,no,yes,accu chek device,HEALTH,resolved by guideline
14,7bc5703237e8,finance,no,yes,my closed account; Financial Industry Ombudsman,FINANCE; FINANCE,resolved by guideline
15,f6d5959a2d4c,health,no,no,,,agreed
16,b089e098d1d6,health,no,no,,,agreed
17,cf5666f901a5,finance,no,no,,,agreed
18,ca6caf759266,health,no,no,,,agreed
19,a84501229e8b,health,no,no,,,agreed
20,d428327804eb,health,no,no,,,agreed
21,fec8bb8b9569,health,yes,yes,Diagnosed type 2; blood results; Accu-Chek,HEALTH; HEALTH; HEALTH,agreed
22,17ac6b497c7e,health,no,no,,,agreed
23,94e86c9e2af2,health,no,no,,,agreed
24,54b09f434240,health,no,no,,,agreed
25,c8935db47793,finance,no,yes,Westpac,FINANCE,resolved by guideline
26,e363003145a5,finance,no,no,,,agreed
27,42e1f9077155,health,no,no,,,agreed
28,30a55a007874,health,no,no,,,agreed
29,c5ef5899a845,health,yes,yes,blood sugars; glucose levels,HEALTH; HEALTH,agreed
30,df0130d41bb1,finance,no,no,,,agreed
31,29c084378bc0,finance,no,no,,,agreed
32,4936243f8bec,health,no,no,,,agreed
33,40e6641fe634,finance,no,no,,,agreed
34,e5fee1300e24,finance,no,no,,,agreed
35,74385bfd9164,finance,yes,yes,19% interest; a loan,FINANCE; FINANCE,agreed
36,ddef1bc95f5e,health,yes,no,,,resolved by guideline
37,982c027827cc,health,no,no,,,agreed
38,e10b844f9249,finance,no,no,,,agreed
39,feddd740b2e0,finance,no,no,,,agreed
40,093fcb5a57f6,finance,yes,yes,my account; my money; 3 days,FINANCE; FINANCE; FINANCE,agreed
41,21b1e36b3eb2,health,no,no,,,agreed
42,1b5981ba46b5,health,no,no,,,agreed
43,401281108784,finance,no,no,,,agreed
44,e1eb1f0039c2,finance,no,no,,,agreed
45,0587c298c9a0,finance,no,no,,,agreed
46,95ad57b7b7da,finance,no,no,,,agreed
47,53ec7a4f522f,finance,no,no,,,agreed
48,cecd0eb02063,health,no,no,,,agreed
49,5aca4106e005,finance,no,no,,,agreed
50,5864119ce5e5,health,no,no,,,agreed
51,254a31c63e7c,health,no,no,,,agreed
52,f6f9dc5e7c1b,finance,no,no,,,agreed
53,b11b9ea510aa,health,no,no,,,agreed
54,848637ca25e1,finance,no,no,,,agreed
55,2d8131e500c8,health,no,no,,,agreed
56,9f8a30e5e557,finance,no,no,,,agreed
57,5f31234dd849,health,no,no,,,agreed
58,d0aabf97362f,finance,no,no,,,agreed
59,a11284447543,health,no,no,,,agreed
60,dcad8c8dcf60,finance,no,no,,,agreed
61,f6e05f745477,finance,no,no,,,agreed
62,dc0a6eeb9d8e,finance,no,no,,,agreed
63,ca11c01c959b,finance,no,no,,,agreed
64,57e6cdf16441,health,no,no,,,agreed
65,850a649de8f9,health,no,no,,,agreed
66,73e11995246d,health,no,no,,,agreed
67,45ca35406b41,finance,no,no,,,agreed
68,0f053c1c834e,finance,no,no,,,agreed
69,65e70a65125f,finance,no,no,,,agreed
70,ee0e1535100b,health,no,no,,,agreed
71,513b06c8dac1,health,no,no,,,agreed
72,e783e28e7821,finance,no,no,,,agreed
73,e16a1cb237b9,health,no,no,,,agreed
74,5a00be712216,health,no,no,,,agreed
75,fbd9b9a72c5a,finance,no,no,,,agreed
76,89a41831cf14,health,no,no,,,agreed
77,bd823a5106ec,health,no,no,,,agreed
78,0f8db9b48a90,finance,no,no,,,agreed
79,6a90ce5b020e,health,no,no,,,agreed
80,4ad90394b35f,finance,no,yes,mortgage,FINANCE,resolved by guideline
81,e52db57a7adc,finance,no,no,,,agreed
82,6f2e40e4cd5d,finance,no,no,,,agreed
83,32c35d3d582a,finance,no,no,,,agreed
84,c30cc9b61f33,health,no,no,,,agreed
85,26b2d8a13499,health,yes,no,,,resolved by guideline
86,05a118cfa9ed,finance,yes,no,,,resolved by guideline
87,d7c09d9ec163,health,no,no,,,agreed
88,32bc6894b95b,health,yes,no,,,resolved by guideline
89,332de3edee79,finance,no,no,,,agreed
90,7983bbc472fb,health,no,no,,,agreed
91,f5cff08bdf86,finance,no,no,,,agreed
92,3fc2788b42b8,finance,no,no,,,agreed
93,79528fb32976,finance,no,no,,,agreed
94,35f757c1aadf,health,no,yes,provider I've seen virtually; session,HEALTH; HEALTH,resolved by guideline
95,d8a505bcb730,finance,no,no,,,agreed
96,d9eec2e813ca,health,no,no,,,agreed
97,4d0d665a9da6,health,no,no,,,agreed
98,1a474530ebaf,finance,no,no,,,agreed
99,5cad9127a067,health,yes,yes,multiple medications; total knee replacement; daily vitamins,HEALTH; HEALTH; HEALTH,agreed
100,9aa861a07b54,finance,no,no,,,agreed

In [ ]:
# Step 1 · Inputs (about 3 minutes)
DATA = f"{BASE}/data"


def rows(path):
    with open(path, encoding="utf-8") as f:
        return sum(1 for _ in f) - 1


if not os.path.exists(f"{DATA}/SST-2/dev.tsv"):
    !cd {DATA} && wget -q -O SST-2.zip https://dl.fbaipublicfiles.com/glue/data/SST-2.zip && unzip -o -q SST-2.zip && rm SST-2.zip
if not os.path.exists(f"{DATA}/QNLI/dev.tsv"):
    !cd {DATA} && wget -q -O QNLI.zip https://dl.fbaipublicfiles.com/glue/data/QNLIv2.zip && unzip -o -q QNLI.zip && rm QNLI.zip
print(f"SST-2: train {rows(f'{DATA}/SST-2/train.tsv'):,} | dev {rows(f'{DATA}/SST-2/dev.tsv'):,}")
print(f"QNLI : train {rows(f'{DATA}/QNLI/train.tsv'):,} | dev {rows(f'{DATA}/QNLI/dev.tsv'):,}")

# Your review dataset (9 Oct), the sampled annotation sheet and the cleaned reviews (for looking up the texts)
def find_reviews(old):
    """The review dataset's folder: the expected place, else a search of My Drive (e.g. a shortcut or an upload)."""
    if os.path.exists(f"{old}/new_data/app_reviews/dev.tsv"):
        return f"{old}/new_data/app_reviews"
    pats = ["/content/drive/MyDrive/*/new_data/app_reviews/dev.tsv", "/content/drive/MyDrive/*/*/new_data/app_reviews/dev.tsv",
            "/content/drive/MyDrive/app_reviews/dev.tsv", "/content/drive/MyDrive/*/app_reviews/dev.tsv"]
    hits = sorted(h for p in pats for h in glob.glob(p))
    return os.path.dirname(hits[0]) if hits else None


REV_SRC = find_reviews(OLD)
if REV_SRC and REV_SRC.endswith("/new_data/app_reviews"):
    OLD = REV_SRC[: -len("/new_data/app_reviews")]   # the 9 Oct folder, wherever it is
print("review dataset folder:", REV_SRC or "NOT FOUND")
RV = f"{DATA}/app_reviews"
os.makedirs(RV, exist_ok=True)
for src, dst in ((f"{REV_SRC}/dev.tsv", f"{RV}/dev.tsv"),
                 (f"{REV_SRC}/train.tsv", f"{RV}/train.tsv"),
                 (f"{REV_SRC}/reviews_clean.jsonl", f"{RV}/reviews_clean.jsonl"),
                 (f"{REV_SRC}/annotation/annotation_sheet.csv", f"{RV}/annotation_sheet.csv")):
    if REV_SRC and os.path.exists(src) and not os.path.exists(dst):
        shutil.copy2(src, dst)
print("reviews:", {f: os.path.exists(f"{RV}/{f}") for f in ("dev.tsv", "reviews_clean.jsonl", "annotation_sheet.csv")})

# Dreaddit, converted exactly as in notebook A/B
DD = f"{DATA}/dreaddit"
os.makedirs(DD, exist_ok=True)
if not os.path.exists(f"{DD}/dev.tsv"):
    import csv, urllib.request
    try:
        if not glob.glob(f"{DD}/**/dreaddit-test.csv", recursive=True):
            urllib.request.urlretrieve("http://www.cs.columbia.edu/~eturcan/data/dreaddit.zip", f"{DD}/dreaddit.zip")
            with zipfile.ZipFile(f"{DD}/dreaddit.zip") as z:
                z.extractall(DD)
        for split, name in (("train", "dreaddit-train.csv"), ("dev", "dreaddit-test.csv")):
            src = glob.glob(f"{DD}/**/{name}", recursive=True)[0]
            with open(src, encoding="utf-8", newline="") as f:
                keep = [r for r in csv.DictReader(f) if r.get("text", "").strip() and r.get("label", "").strip() in ("0", "1")]
            with open(f"{DD}/{split}.tsv", "w", encoding="utf-8") as f:
                f.write("sentence\tlabel\n")
                for r in keep:
                    f.write(" ".join(r["text"].split()) + "\t" + r["label"].strip() + "\n")
    except Exception as e:
        print("!! Dreaddit download failed; the judge skips Dreaddit:", e)
print("Dreaddit dev:", rows(f"{DD}/dev.tsv") if os.path.exists(f"{DD}/dev.tsv") else "missing")

# Notebook A/B's outputs
SAN_AB = f"{NEW}/san"
found = sorted(glob.glob(f"{SAN_AB}/*/*/seed_42/eps_*/**/dev.tsv", recursive=True))
print(f"A/B sanitized dev files (seed 42): {len(found)}")
if not found:
    print("!! None found. Run notebook A/B first (or wait for its Tier 1 to finish); the judge needs them.")
UNSAN_RUN = next((p for p in (f"{NEW}/results/sst2__unsanitized__seed42", f"{OLD}/results/SST-2_unsanitized")
                  if os.path.exists(f"{p}/metrics.json")), None)
print("unsanitized SST-2 baseline:", UNSAN_RUN or "missing")
GLOVE_FILTERED = f"{NEW}/data/glove.filtered.txt"
print("filtered GloVe from A/B:", os.path.exists(GLOVE_FILTERED))
OLD_SST2 = f"{OLD}/SanText/output_SanText_glove/SST-2"
print("9 Oct sanitized SST-2:", sorted(os.path.basename(p) for p in glob.glob(f"{OLD_SST2}/eps_*")))

In [ ]:
# Step 2 · Original datasets, measured from the files (SST-2 and QNLI; about 3 minutes)
import collections, statistics
sys.path.insert(0, f"{BASE}/SanText")
from utils import get_vocab_SST2, get_vocab_QNLI   # the authors' vocabulary functions
from spacy.lang.en import English

if not os.path.exists(f"{BASE}/results_c/datasets.json"):
    tok = English()
    stats = {}
    for name, folder, fmt in (("SST-2", f"{DATA}/SST-2", "single"), ("QNLI", f"{DATA}/QNLI", "pair")):
        d = {}
        for split in ("train", "dev", "test"):
            path = f"{folder}/{split}.tsv"
            if not os.path.exists(path):
                continue
            with open(path, encoding="utf-8") as f:
                header = f.readline().rstrip("\n")
                lines = [l.rstrip("\n").split("\t") for l in f]
            d[f"{split}_rows"] = len(lines)
            if split == "train":
                d["header"] = header
                d["examples"] = ["\t".join(l) for l in lines[:2]]
            if split == "dev":
                d["examples"] = d.get("examples", []) + ["\t".join(l) for l in lines[:1]]
            if split != "test":
                labels = collections.Counter(l[-1] for l in lines)
                d[f"{split}_labels"] = dict(labels)
                if fmt == "single":
                    d[f"{split}_words_mean"] = round(statistics.mean(len(l[0].split()) for l in lines), 1)
                else:
                    d[f"{split}_question_words_mean"] = round(statistics.mean(len(l[1].split()) for l in lines), 1)
                    d[f"{split}_sentence_words_mean"] = round(statistics.mean(len(l[2].split()) for l in lines), 1)
        vocab = (get_vocab_SST2 if fmt == "single" else get_vocab_QNLI)(folder, tok, tokenizer_type="word")
        d["vocabulary_authors_construction"] = len(vocab)
        stats[name] = d
        print(name, json.dumps({k: v for k, v in d.items() if k != "examples"}))
    json.dump(stats, open(f"{BASE}/results_c/datasets.json", "w"), indent=2, ensure_ascii=False)
    backup(f"{BASE}/results_c/datasets.json")
else:
    print("already done:", json.load(open(f"{BASE}/results_c/datasets.json")).keys())

# QNLI with GloVe, block by block, in the background (CPU; the GPU steps continue meanwhile)
QNLI_OUT = f"{BASE}/san_qnli/QNLI/eps_3.00/meta.json"
qnli_proc = None
if not os.path.exists(QNLI_OUT):
    glove_full = f"{DATA}/glove.840B.300d.txt"
    cmd = (f"([ -f {glove_full} ] || ((wget -q -O {DATA}/glove.zip https://huggingface.co/stanfordnlp/glove/resolve/main/glove.840B.300d.zip"
           f" || wget -q -O {DATA}/glove.zip https://nlp.stanford.edu/data/glove.840B.300d.zip) && unzip -o -q {DATA}/glove.zip -d {DATA} && rm -f {DATA}/glove.zip))"
           f" && {sys.executable} -u santext_chunked.py --task QNLI --embedding glove --glove {glove_full} --epsilon 3"
           f" --data_dir {DATA}/QNLI --output_dir {BASE}/san_qnli/QNLI --santext_dir {BASE}/SanText --splits train,dev")
    qnli_proc = subprocess.Popen(["bash", "-c", cmd], cwd=BASE, stdout=open(f"{BASE}/logs/qnli_background.log", "w"),
                                 stderr=subprocess.STDOUT)
    print("QNLI sanitization started in the background (log: logs/qnli_background.log)")

In [ ]:
# Step 3 · LLM judge (about 25 minutes, mostly unattended)
J = f"{BASE}/results_c/judge"
cfg = {"labels": f"{BASE}/annotation_labels.csv", "reviews_clean": f"{RV}/reviews_clean.jsonl",
       "annotation_sheet": f"{RV}/annotation_sheet.csv", "san_root": SAN_AB, "old_sst2": OLD_SST2,
       "original_dev": {"sst2": f"{DATA}/SST-2/dev.tsv", "dreaddit": f"{DD}/dev.tsv" if os.path.exists(f"{DD}/dev.tsv") else None,
                        "reviews": f"{RV}/dev.tsv" if os.path.exists(f"{RV}/dev.tsv") else None}}
json.dump(cfg, open(f"{J}/judge_inputs.json", "w"), indent=2)
# items are rebuilt every time (cheap); the judge skips items it has already rated
run([sys.executable, "-u", "build_judge_items.py", "--config", f"{J}/judge_inputs.json", "--out_dir", J], "judge_items.log")
backup(f"{J}/human_check.xlsx", f"{J}/human_check_key.csv", f"{J}/items_report.json")
print(open(f"{J}/items_report.json").read() if os.path.exists(f"{J}/items_report.json") else "no items")
if os.path.exists(f"{J}/judge_items.jsonl"):
    run([sys.executable, "-u", "llm_judge.py", "--items", f"{J}/judge_items.jsonl", "--out", f"{J}/judge_results.jsonl"],
        "llm_judge.log")
    backup(f"{J}/judge_results.jsonl")

In [ ]:
# Step 4 · SanText with BERT embeddings on SST-2, block by block (about 5 minutes)
BERT_EPS = [8.0, 12.0, 14.0, 16.0, 20.0]
for e in BERT_EPS:
    if os.path.exists(f"{BASE}/san_bert/SST-2/eps_{e:.2f}/meta.json"):
        print(f"already done: eps {e:g}")
        continue
    run([sys.executable, "-u", "santext_chunked.py", "--task", "SST-2", "--embedding", "bert", "--epsilon", str(e),
         "--seed", "42", "--data_dir", f"{DATA}/SST-2", "--output_dir", f"{BASE}/san_bert/SST-2",
         "--santext_dir", f"{BASE}/SanText"] + (["--check_rows", "300"] if e == 14.0 else []), f"san_bert_eps{e:g}.log")
    backup(f"{BASE}/san_bert/SST-2/eps_{e:.2f}")

In [ ]:
# Step 5 · Mask token inference attack: unsanitized, then each epsilon; as released and fixed (about 5 minutes)
A = f"{BASE}/results_c/attack"
targets = [("unsanitized", "none")] + [(f"eps{e:g}", f"{BASE}/san_bert/SST-2/eps_{e:.2f}/dev.tsv") for e in BERT_EPS]
for name, san in targets:
    if os.path.exists(f"{A}/{name}.json"):
        print("already done:", name)
        continue
    if san != "none" and not os.path.exists(san):
        print("missing sanitized file, skipped:", name)
        continue
    run([sys.executable, "-u", "mask_attack.py", "--original", f"{DATA}/SST-2/dev.tsv", "--sanitized", san,
         "--name", name, "--out_dir", A], f"attack_{name}.log")
    backup(f"{A}/{name}.json")

In [ ]:
# Step 6 · Does the block-wise sanitizer match the authors' code? GloVe, SST-2, eps = 1, 2, 3 (about 5 minutes)
if os.path.exists(GLOVE_FILTERED):
    for e in (1.0, 2.0, 3.0):
        if os.path.exists(f"{BASE}/san_glove_check/SST-2/eps_{e:.2f}/meta.json"):
            print(f"already done: eps {e:g}")
            continue
        run([sys.executable, "-u", "santext_chunked.py", "--task", "SST-2", "--embedding", "glove", "--glove", GLOVE_FILTERED,
             "--epsilon", str(e), "--seed", "42", "--data_dir", f"{DATA}/SST-2", "--output_dir", f"{BASE}/san_glove_check/SST-2",
             "--santext_dir", f"{BASE}/SanText", "--splits", "dev", "--check_rows", "2000"], f"glove_check_eps{e:g}.log")
        backup(f"{BASE}/san_glove_check/SST-2/eps_{e:.2f}")
else:
    print("!! The filtered GloVe file from notebook A/B isn't on Drive yet; this check is skipped.")

In [ ]:
# Step 7 · Fine-tuning on the BERT-sanitized SST-2 at epsilon = 16, the point of the paper's privacy claim (Table 3: 0.8698; about 13 minutes)
FT_EPS = (16.0,)   # add 12.0, 14.0 (paper: 0.6084, 0.7548) if you have 30 spare minutes
for e in FT_EPS:
    name = f"sst2__SanText_bert__eps{e:.2f}__seed42"
    if os.path.exists(f"{BASE}/results/{name}/metrics.json"):
        print("already done:", name)
        continue
    if not os.path.exists(f"{BASE}/san_bert/SST-2/eps_{e:.2f}/train.tsv"):
        print("missing sanitized data, skipped:", name)
        continue
    run([sys.executable, "-u", "finetune_sanitized.py", "--data_dir", f"{BASE}/san_bert/SST-2/eps_{e:.2f}",
         "--run_name", name, "--fp16", "--log_every", "300", "--seed", "42"], f"ft_{name}.log")
    backup(f"{BASE}/results/{name}")

In [ ]:
# Step 8 · Summary tables and one zip to upload (about 2 minutes)
if qnli_proc is not None and qnli_proc.poll() is None:
    print("waiting up to 15 minutes for the QNLI background job ...", flush=True)
    try:
        qnli_proc.wait(timeout=900)
    except subprocess.TimeoutExpired:
        print("QNLI is still running; the summary leaves it out (re-run this cell later to include it).")
print("QNLI:", open(f"{BASE}/logs/qnli_background.log", errors="ignore").read().splitlines()[-3:]
      if os.path.exists(f"{BASE}/logs/qnli_background.log") else "not started")
backup(f"{BASE}/san_qnli/QNLI/eps_3.00/meta.json", f"{BASE}/logs/qnli_background.log")

plan_c = {"results_dir": f"{BASE}/results", "unsanitized_run": "_unsan", "bert_dir": f"{BASE}/san_bert/SST-2",
          "bert_eps": BERT_EPS, "glove_check_dir": f"{BASE}/san_glove_check/SST-2", "glove_check_eps": [1.0, 2.0, 3.0],
          "sst2_dev": f"{DATA}/SST-2/dev.tsv",
          "authors_glove_dirs": {"9 Oct": OLD_SST2, "notebook A/B": f"{SAN_AB}/sst2/SanText/seed_42"}}
if UNSAN_RUN:
    shutil.copytree(UNSAN_RUN, f"{BASE}/results/_unsan", dirs_exist_ok=True)
json.dump(plan_c, open(f"{BASE}/results_c/plan_c.json", "w"), indent=2)
!cd {BASE} && python -u analyse_c.py --root results_c --plan_c results_c/plan_c.json

files = glob.glob("results_c/**/*", recursive=True) + glob.glob("logs/*")
files += glob.glob("results/*/metrics.json") + glob.glob("results/*/dev_predictions.tsv")
files += glob.glob("san_bert/SST-2/eps_*/meta.json") + glob.glob("san_bert/SST-2/eps_*/dev.tsv")
files += glob.glob("san_glove_check/SST-2/eps_*/meta.json") + glob.glob("san_qnli/QNLI/eps_*/meta.json")
with zipfile.ZipFile(f"{BASE}/notebook_c_outputs.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(set(files)):
        if os.path.isfile(p) and not p.endswith("judge_items.jsonl"):
            z.write(p)
print(f"\nnotebook_c_outputs.zip: {os.path.getsize(f'{BASE}/notebook_c_outputs.zip') / 1e6:.1f} MB")
backup(f"{BASE}/notebook_c_outputs.zip", f"{BASE}/results_c")
from google.colab import files as colab_files
for p in (f"{BASE}/notebook_c_outputs.zip", f"{BASE}/results_c/judge/human_check.xlsx"):
    if os.path.exists(p):
        colab_files.download(p)

### Next
1. Upload `notebook_c_outputs.zip` in the chat.
2. Open `human_check.xlsx` (it's also in `MyDrive/COMP8240_final/c/results_c/judge/`), fill the **your answer** column (about 20 minutes, instructions on the first sheet), and upload it. Don't look at the judge's answers first: the check only counts if it's blind.